## Setup

In [4]:
import Pkg
Pkg.activate(@__DIR__, io=devnull)
using HybridSchurLab, IncompleteLU, LinearAlgebra, BenchmarkTools, Krylov;

In [2]:
# Parameters
n_parts = 64
n_grid = 1000

# 1. Generate & Partition Matrix
A_raw = create_laplacian_2d(n_grid)
perm, int_idxs, g_idxs = get_partition_indices_laplacian_FD(n_grid, n_parts)
A_p = A_raw[perm, perm]

# 2. Setup Schur Operator
op = setup_schur_operator(A_p, int_idxs, g_idxs)
S = as_linear_map(op)

# 3. Setup RHS
b_p = ones(size(A_p, 1))
b_I = b_p[vcat(int_idxs...)]
b_g = b_p[g_idxs]

# Compute modified RHS: g = b_g - A_ΓI * (A_II \ b_I)
g = b_g - op.A_gamma_I * solve_interior(op, b_I)

println("Schur Product benchmark:")
@btime mul!(similar($g), $S, $g);

Schur Product benchmark:
  11.766 ms (4 allocations: 248.15 KiB)


## Mixed Precision

In [ ]:
# Setup Preconditioner Float64
prec64 = setup_2lp(A_p, int_idxs, g_idxs, op, Float64)
println("type of LU factorizations: ", typeof(prec64.local_LUs[1]))

println("Preconditioner benchmark:")
y_bench = similar(g)
@btime ldiv!($y_bench, $prec64, $g);

type of LU factorizations: IncompleteLU.ILUFactorization{Float64, Int64}
Preconditioner benchmark:
  5.781 ms (1 allocation: 128 bytes)


In [ ]:
# Setup Preconditioner Float32
prec = setup_2lp(A_p, int_idxs, g_idxs, op, Float32)
println("type of LU factorizations: ", typeof(prec.local_LUs[1]))

println("Preconditioner benchmark:")
y_bench = similar(g)
@btime ldiv!($y_bench, $prec, $g);

type of LU factorizations: IncompleteLU.ILUFactorization{Float32, Int64}
Preconditioner benchmark:
  3.921 ms (1 allocation: 128 bytes)


## Solve using CG

In [20]:
# 1. Solve Interface via CG
workspace_S = CgWorkspace(S, g)
t_hybrid = @belapsed Krylov.cg!($workspace_S, $S, $g; M=$prec, ldiv=true, itmax=1000, rtol=1e-12)

x_interface = workspace_S.x
println("Interface solve finished in $(workspace_S.stats.niter) iterations.")

# 2. Reconstruct Interior: x_I = A_II \ (b_I - A_I_gamma * x_interface)
mul!(op.tmp1, op.A_I_gamma, x_interface)
b_tmp = b_I .- op.tmp1
x_I = solve_interior(op, b_tmp)

# 3. Full solution and Verification
x_full = vcat(x_I, x_interface)
residual_norm = norm(A_p * x_full - b_p)
println("Final Residual Norm: $residual_norm")

Interface solve finished in 192 iterations.
Final Residual Norm: 8.738182125939273e-8


In [21]:
# --- Method 1: Direct Solve ---
t_direct = @belapsed $A_p \ $b_p
x_direct = A_p \ b_p
res_direct = norm(A_p * x_direct - b_p)

# --- Method 2: CG with ILU(0.1) on full A ---
LU_fact = ilu(A_p, τ=0.1)

function run_ilu_cg(A, b, prec)
    w = CgWorkspace(A, b)
    Krylov.cg!(w, A, b; M=prec, ldiv=true, rtol=1e-12)
    return w
end

t_ilu = @belapsed run_ilu_cg($A_p, $b_p, $LU_fact)
w_ilu = run_ilu_cg(A_p, b_p, LU_fact)
res_ilu = norm(A_p * w_ilu.x - b_p)

# --- Results Table ---
comparison_data = [
    "Solver"       "Iterations"           "Time (s)"   "Residual" ;
    "Hybrid Schur" workspace_S.stats.niter t_hybrid    residual_norm ;
    "CG + ILU"     w_ilu.stats.niter       t_ilu       res_ilu ;
    "Direct"       1                       t_direct    res_direct
]

display(comparison_data)

4×4 Matrix{Any}:
 "Solver"           "Iterations"   "Time (s)"   "Residual"
 "Hybrid Schur"  192              3.43948      8.73818e-8
 "CG + ILU"      511              8.02696      1.93738e-7
 "Direct"          1              1.24815      1.92487e-8